# 03 - Feature Engineering and Pre-Processing

##### 03-00 General Model Plan

Feature Engineering

- Create `duration_days` from the cleaned date columns
- Remove invalid and outlier duration rows
- Create the target variable from duration ranges
- Save the final cleaned modeling dataset

Pre-Processing

- Split dataset into training and testing subsets
- Split pre-processing steps into 3 groups (Text, Categories, and Words/Lengths)
- Use TF-IDF vectorization to convert text values (summaries and descriptions) to numeric matrices
- Use One Hot Encoder to encode catgories into numeric values with no relations, e.g, Bug and Improvement are completely unrelated
- Use Standard Scaler to standardizes the numeric values

Model training

- Use Logistic Regression


## 03-01 Importing CSV file and defining the target

Before we start training the model using the cleaned dataset, we need to create the target value from the cleaned date columns.

In [1]:
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split

jira_df = pd.read_csv("../data/processed/jira_issues_cleaned.csv", index_col=0)

task_df = jira_df.copy()

for col in ["created", "resolutiondate"]:
    task_df[col] = pd.to_datetime(task_df[col])

task_df.head()

,summary,description,priority_name,issuetype_name,summary_char_count,summary_word_count,description_char_count,description_word_count,created,resolutiondate
0,Update config browser to work with the new syntax,The config browser used Velocity calling the t...,Minor,Improvement,49,9,176,28,2005-01-01 07:47:50,2005-01-01 07:50:46
1,XALAN_C 1.9 or current do not build on Fedora ...,Two types of errors: 1- runConfigure and confi...,Blocker,Bug,52,11,3445,206,2004-12-25 22:50:30,2004-12-30 05:30:36
2,"Problem with ADD new post, and DELETE post.","When trying to add new post, I was getting nex...",Critical,Bug,43,8,2560,124,2005-01-01 13:52:46,2005-01-02 15:21:00
5,XmlConfigurator should support namespaces othe...,"Currently, the XmlConfigurator assumes that th...",Major,Improvement,66,7,848,138,2004-12-27 01:34:17,2005-01-03 10:21:28
6,GroovyServlet will crash if parameters are pas...,If parameters are being passed to a groovy ser...,Major,Bug,65,10,419,65,2004-12-28 18:50:52,2005-01-03 11:29:27


## 03-02 Create duration days derived variable

Calculate task duration in days.

This target is derived from:

`duration_days = resolutiondate - created`

In [3]:
task_df["duration_days"] = (task_df["resolutiondate"] - task_df["created"]).dt.total_seconds() / (60 * 60 * 24)

task_df["duration_days"].describe()

count    938519.000000
mean        201.985744
std         517.790205
min           0.000000
25%           1.042581
50%          11.802130
75%         110.378437
max        8001.517257
Name: duration_days, dtype: float64

The above statistics show target-distribution issues that need to be handled before modeling:

- The target has a strong long-tail pattern.
- Extreme outliers pull the mean far above the median.

#### Inspect long-tail durations

Before removing long-running tasks, first inspect how many rows would be affected.

In [4]:
display(task_df["duration_days"].describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]))

duration_over_365_count = (task_df["duration_days"] > 90).sum()
print(f"Rows with duration_days > 90: {duration_over_365_count:,}")

count    938519.000000
mean        201.985744
std         517.790205
min           0.000000
50%          11.802130
75%         110.378437
90%         617.085472
95%        1184.935810
99%        2689.762594
max        8001.517257
Name: duration_days, dtype: float64

Rows with duration_days > 90: 253,611


Removing this many rows is a significant change, although the dataset may still contain enough records for modeling.

Tasks that exceed 90 days may be signs of:

- stale issues
- abandoned tickets
- imported/migrated issues
- tickets closed administratively
- issues left open for months/years without active work
- old backlog cleanup

These rows may not represent normal task-completion behavior, so removing or capping them can make the target easier for the model to learn.

In [5]:
task_df = task_df[task_df["duration_days"] <= 90].copy()

display(task_df.shape)
task_df.describe()

(684908, 11)

,summary_char_count,summary_word_count,description_char_count,description_word_count,created,resolutiondate,duration_days
count,684908.000000,684908.000000,6.849080e+05,684908.000000,684908,684908,684908.000000
mean,57.157488,7.904365,9.118984e+02,79.726051,2015-11-02 09:48:32.312944,2015-11-15 20:09:41.338952,13.431354
min,0.000000,0.000000,0.000000e+00,0.000000,2002-05-10 18:37:38,2002-05-10 18:54:17,0.000000
25%,40.000000,5.000000,1.170000e+02,16.000000,2012-06-29 23:15:46.250000,2012-07-11 18:46:10.250000,0.451236
50%,54.000000,7.000000,2.870000e+02,40.000000,2016-02-08 16:27:00,2016-02-20 13:51:43.500000,3.872697
75%,70.000000,10.000000,6.780000e+02,84.000000,2019-08-20 23:11:53.500000,2019-09-04 17:42:22,17.181183
max,255.000000,48.000000,7.615811e+06,351933.000000,2024-11-06 14:09:26,2025-01-27 21:10:15,89.999618
std,23.981032,3.657953,1.197618e+04,612.868118,NaN,NaN,20.267580


Tasks resolved in under 2 hours may represent very small fixes, administrative updates, or tickets closed immediately after creation. Removing them can reduce short-duration skew, but this should be treated as a modeling decision rather than a universal rule.

In [6]:
task_df = task_df[task_df["duration_days"] >= (2/24)].copy()

## 03-03 Create duration classification target

Create a classification target from `duration_days`.

The goal is to choose ranges that are:

- simple enough to explain
- not overly skewed toward one class
- aligned with what the task summaries and descriptions appear to represent

The earlier EDA showed two important cleanup decisions before classification:

- remove tasks completed in under 2 hours
- remove very long tasks above the selected long-tail cutoff

After those removals, compare a few possible class ranges before choosing the final classes.

### Duration ranges

- **Very Quick:** 2 hours to 1 day
- **Quick:** greater than 1 day and up to 3 days
- **Standard:** greater than 3 days and up to 7 days
- **Extended:** greater than 7 days and up to 30 days
- **Long-term:** greater than 30 days and up to 90 days


In [7]:
def duration_category(days):
    if days <= 1:
        return "Very Quick"
    if days <= 3:
        return "Quick"
    if days <= 7:
        return "Standard"
    if days <= 30:
        return "Extended"
    return "Long-term"

duration_order = ["Very Quick", "Quick", "Standard", "Extended", "Long-term"]

task_df["duration_category"] = task_df["duration_days"].apply(duration_category)

class_counts = task_df["duration_category"].value_counts().reindex(duration_order)
class_percentages = (task_df["duration_category"].value_counts(normalize=True).reindex(duration_order).mul(100).round(2))

duration_class_summary = pd.DataFrame({
    "count": class_counts,
    "percent": class_percentages,
})

display(task_df.shape)
display(task_df.describe())

duration_class_summary

(579005, 12)

,summary_char_count,summary_word_count,description_char_count,description_word_count,created,resolutiondate,duration_days
count,579005.000000,579005.000000,5.790050e+05,579005.000000,579005,579005,579005.000000
mean,57.439276,7.927838,9.653918e+02,84.190903,2015-12-26 03:11:54.000680,2016-01-11 00:24:48.646678,15.883966
min,1.000000,1.000000,0.000000e+00,0.000000,2002-05-10 18:37:38,2002-06-03 18:35:07,0.083333
25%,41.000000,5.000000,1.290000e+02,18.000000,2012-09-03 13:53:35,2012-09-18 00:29:07,1.240475
50%,54.000000,7.000000,3.100000e+02,43.000000,2016-04-29 02:36:11,2016-05-13 23:50:41,6.054028
75%,70.000000,10.000000,7.260000e+02,89.000000,2019-11-07 23:28:40,2019-11-22 07:37:16,21.901597
max,255.000000,48.000000,7.615811e+06,351933.000000,2024-11-06 12:43:08,2025-01-27 21:10:15,89.999618
std,23.967963,3.660774,1.278004e+04,648.832244,NaN,NaN,21.142487


,count,percent
duration_category,,
Very Quick,123476,21.33
Quick,90029,15.55
Standard,94631,16.34
Extended,160206,27.67
Long-term,110663,19.11


### 03-03.1 Spot-check task text by class

Sample a few summaries and descriptions from each final class. This is a manual sanity check that the classes are not obviously misleading for the type of task text the model will receive.

In [8]:
sample_columns = [
    "summary",
    "description",
    "priority_name",
    "issuetype_name",
    "duration_days",
    "duration_category",
]

for category in duration_order:
    print(category)
    class_sample = task_df.loc[
        task_df["duration_category"] == category,
        sample_columns,
    ].sample(n=5, random_state=42)
    display(class_sample)

Very Quick


,summary,description,priority_name,issuetype_name,duration_days,duration_category
304387,S3 VFS driver shows size warning,The message printed in the log is: {code:java}...,Trivial,Bug,0.103542,Very Quick
20431,Use JNDI provided by the container,Right now we're using our own lightweight JNDI...,Major,Improvement,0.199444,Very Quick
896555,Unable to remove secondary ips though there ar...,Steps to reproduce : 1. Have at least 1 VM dep...,Critical,Bug,0.668715,Very Quick
249860,DM does not support java9+,Dependency Manager can't be built/used on java...,Major,Improvement,0.282870,Very Quick
1059743,Implement a different version of Naive Bayes P...,There has been many dependency issues with the...,Major,Improvement,0.611157,Very Quick


Quick


,summary,description,priority_name,issuetype_name,duration_days,duration_category
23876,Update for some outdated language files,Update for CommonUiLabels_ru.properties. Ukrai...,Trivial,Improvement,2.736273,Quick
330445,MatrixBlock equals,NaN,Major,Improvement,1.117789,Quick
27032,[drlvm][verifier] cleaning edges from unreacha...,"When we clean an unreachable node, all incomin...",Major,Sub-task,2.017975,Quick
968959,hbase1.3.1 regionerver Crash (bucketcache),"hbase 1.3.1 regionserver crash , configure buc...",Major,Bug,2.872569,Quick
850688,Incorrect results or NPE when inserting null v...,Example: {noformat} create or replace temp vie...,Major,Bug,2.313519,Quick


Standard


,summary,description,priority_name,issuetype_name,duration_days,duration_category
929996,Hovering over a border of a datagrid header cr...,Steps to reproduce: 1. Run bug app 2. Move mou...,Major,Bug,5.154167,Standard
1125696,unix_timestamp runtime crash,in some cases select unix_timestamp('2017-11-0...,Major,Bug,6.487141,Standard
743420,Improve the Adapter API to directly yield Geom...,"Currently, the Adapter in Sedona SQL toDf meth...",Major,Improvement,3.138241,Standard
77173,TestHdfsProxy fails on 0.20,TestHdfsProxy fails with the following excepti...,Major,Bug,3.168345,Standard
265330,HTML redirections are not followed when using ...,Html redirections using meta tags are supporte...,Major,Bug,3.828900,Standard


Extended


,summary,description,priority_name,issuetype_name,duration_days,duration_category
776228,RecordsOut metric for sinks is inaccurate,"Currently, the metric is computed on the opera...",Major,Improvement,10.003588,Extended
272072,Support defining arbitrary autoscaling simulat...,In many cases where the {{bin/solr autoscaling...,Major,Improvement,14.284792,Extended
619987,uv3 migration tool should add comment about th...,The current v3 migration tool should add a com...,Minor,Improvement,16.079907,Extended
824509,Download-Link on main page is wrong,On [https://logging.apache.org/chainsaw/2.x/] ...,Major,Improvement,26.156331,Extended
91561,Implement encodeBookmarkableURL and encodeRedi...,Methods for encodeBookmarkableURL and encodeRe...,Minor,Task,21.176968,Extended


Long-term


,summary,description,priority_name,issuetype_name,duration_days,duration_category
14696,ExtensionsFilter calls addResource.parseRespon...,ExtensionsFilter does not check the content ty...,Major,Bug,34.061921,Long-term
320797,Make the HBaseMiniCluster compliant with Kerberos,Whne using MiniKDC and the minicluster in a un...,Major,Improvement,57.203241,Long-term
840936,"[C++] Writing float32 values with ""Dictionary ...",If I try to read the attached csv file with py...,Major,Bug,45.838900,Long-term
358553,Zero-sized rows in BufferedTupleStream can cau...,When running the following query on the DEBUG ...,Major,Bug,70.795475,Long-term
930261,WebService unable to read WSDL when it was abl...,Steps to reproduce: 1. Use the attached sample...,Major,Bug,47.926644,Long-term


The samples should not be expected to prove the classes are perfect. They only check that the ranges are reasonable enough for the first text-classification model. Actual performance should still be tested during model training.

## 03-04 Save final cleaned dataset

Save the final cleaned dataset after feature engineering.

In [ ]:
final_cleaned_columns = [
    "summary",
    "description",
    "priority_name",
    "issuetype_name",
    "summary_char_count",
    "summary_word_count",
    "description_char_count",
    "description_word_count",
    "duration_days",
    "duration_category",
]

final_cleaned_df = task_df[final_cleaned_columns].copy()
final_cleaned_df_sample = final_cleaned_df.sample(n=100, random_state=42)

output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

final_cleaned_path = output_dir / "final_cleaned.csv"
final_cleaned_df.to_csv(final_cleaned_path)

final_cleaned_df_sample_path = output_dir / "final_cleaned_sample.csv"
final_cleaned_df_sample.to_csv(final_cleaned_df_sample_path)

print(f"Final cleaned modeling rows: {final_cleaned_df.shape[0]:,}")
print(f"Final cleaned modeling columns: {final_cleaned_df.shape[1]:,}")
print(f"Saved final cleaned CSV file to: {final_cleaned_path}")
print(f"Saved final cleaned sample CSV file to: {final_cleaned_path}")

final_cleaned_df.head()

Final cleaned modeling rows: 579,005
Final cleaned modeling columns: 10
Saved final cleaned CSV file to: ..\data\processed\final_cleaned.csv


,summary,description,priority_name,issuetype_name,summary_char_count,summary_word_count,description_char_count,description_word_count,duration_days,duration_category
1,XALAN_C 1.9 or current do not build on Fedora ...,Two types of errors: 1- runConfigure and confi...,Blocker,Bug,52,11,3445,206,4.277847,Standard
2,"Problem with ADD new post, and DELETE post.","When trying to add new post, I was getting nex...",Critical,Bug,43,8,2560,124,1.061273,Quick
5,XmlConfigurator should support namespaces othe...,"Currently, the XmlConfigurator assumes that th...",Major,Improvement,66,7,848,138,7.366100,Extended
6,GroovyServlet will crash if parameters are pas...,If parameters are being passed to a groovy ser...,Major,Bug,65,10,419,65,5.693461,Standard
8,Groovy / Java Mismatch with simple class,The following code compiles and runs as a java...,Major,Bug,40,7,1438,65,32.141562,Long-term


## 03-05 Split training and test subsets

Before we start training the model using the final cleaned dataset, we need to split the dataset into subsets where each subset is going to be used for different purposes being as follows:

- One subset for training the model
- One subset for testing the model

We're going to remove the `duration_days` column to avoid any data leakage.

Then assign *x* the all cleaned dataframe features EXCEPT the target feature, i.e, `duration_category`.

We assign the *y* the target feature, i.e, `duration_category`.

And finally split the data set into training and testing subsets.

In [ ]:
modeling_df = final_cleaned_df.copy()
modeling_df["total_text"] = (modeling_df["summary"].fillna("") + " " + modeling_df["description"].fillna(""))

modeling_df = modeling_df.drop(columns="duration_days")
x = modeling_df.drop(columns="duration_category")
y = modeling_df["duration_category"]

display(x)
display(y)

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2)

## 03-06 Text feature extraction

#### Text-based feature extraction

As is, computers can't understand human-readable datatypes such as strings or characters. Therefore, we must convert such types into numeric format using what is called **encoding** or **text vectorization**.

Here we tranform the independent features to numeric-matrix values using an TF-IDF vectorizer.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer()

x_train_text = vectorizer.fit_transform(x_train["total_text"])
x_test_text = vectorizer.transform(x_test["total_text"])

#### Category-based feature extraction

Features that show categorical values such as `issuetype._name` and `priority_name` are also text values that need numeric encoding, e.g, Bug or Improvement. 

The categorical values present in these columns aren't necessary related. Therefore, we need to encode these values using nominal encoding methods such as *One Hot Encoding*.

In [ ]:
from sklearn.preprocessing import OneHotEncoder

one_hot_enc = OneHotEncoder()

x_train_cat = one_hot_enc.fit_transform(x_train[["issuetype_name", "priority_name"]])
x_test_cat = one_hot_enc.transform(x_test[["issuetype_name", "priority_name"]])

#### Numeric feature scaling

There exists features that are already represented using numeric valuesm, e.g, summary word count, these features need to be scaled in order to be trained properly.

In [ ]:
from sklearn.preprocessing import StandardScaler

numeric_cols = [
    "summary_char_count",
    "summary_word_count",
    "description_char_count",
    "description_word_count",
]

ss = StandardScaler()

x_train_num = ss.fit_transform(x_train[numeric_cols])
x_test_num = ss.transform(x_test[numeric_cols])

#### Target pre-processing

Next we need to similarly display `duration_categories` as numeric values on a scale where:

- Very Quick -> 0.0
- Quick -> 1.0
- Standard -> 2.0
- Extended -> 3.0
- Long term -> 4.0

This is what is known as *Ordinal Encoding*, it encodes the target value according to a spectrum where values near each other are somewhat related and values farther from each other show the least similarity.

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

target_map = {
    "Very Quick": 0.0,
    "Quick": 1.0,
    "Standard": 2.0,
    "Extended": 3.0,
    "Long-term": 4.0
}

y_train = y_train.map(target_map)
y_test = y_test.map(target_map)

display(y_train)
display(y_test)